# Random Protein Sampling

![Random Protein Sampling](https://proto-bio.github.io/proto-assets/images/tool/random_protein/hero.png)

This notebook demonstrates `run_random_protein_sample`, which generates protein sequence variants by substituting amino acids at masked positions using codon scheme-biased distributions. It covers pre-masked sequences, automatic position selection via masking strategies, and codon scheme comparisons that mirror the amino acid distributions of common degenerate oligonucleotide libraries.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("random_protein")
display_overview("random_protein")
display_docs_section("random_protein", "Background")

# Random Protein Sampling

Random Protein Sampling fills masked positions in protein sequences with random amino acids drawn from a configurable [codon scheme](https://en.wikipedia.org/wiki/Genetic_code). Different codon schemes produce different amino acid frequency distributions, mirroring the biases of real degenerate codon libraries used in experimental directed evolution.

- **Tool key**: `random-protein-sample`
- **Input**: Protein sequences (with optional `_` masks at positions to mutate)
- **Output**: Sequences with masked positions filled by random amino acids
- **Execution**: CPU only, no external dependencies

## Background

**What does this tool do?**
This tool performs random [mutagenesis](https://en.wikipedia.org/wiki/Mutagenesis) at the protein level. Given a protein sequence, it identifies positions to mutate and replaces them with random amino acids sampled according to a codon scheme's frequency distribution.

**Why is this important?**
Random protein mutagenesis is central to [protein engineering](https://en.wikipedia.org/wiki/Protein_engineering):
- **Directed evolution libraries**: Experimental mutagenesis uses degenerate codons (e.g., NNK) that encode amino acids with non-uniform frequencies. Simulating these distributions computationally helps predict library diversity.
- **Baseline comparisons**: Random mutagenesis provides a null model for evaluating whether model-guided designs (ESM2, ProteinMPNN) outperform chance.
- **Combinatorial screening**: Randomizing a few key positions generates focused libraries for functional assays.

**Codon schemes:**
Each scheme represents a degenerate codon pattern used in synthetic biology. The scheme determines which amino acids can appear and their relative frequencies:

| Scheme | Codons | Amino Acids | Use Case |
|--------|--------|-------------|----------|
| `UNIFORM` | N/A | All 20, equal weight | Unbiased random sampling |
| `NNK` | 32 | All 20 | Standard library design, reduced stop codons |
| `NNS` | 32 | All 20 | Alternative to NNK, same coverage |
| `NDT` | 12 | 12 AAs | Zero-redundancy, small focused libraries |
| `DBK` | 18 | 18 AAs | Specialized scheme |
| `NRT` | 8 | 8 AAs | Purine/pyrimidine combination |

In NNK/NNS schemes, amino acids encoded by more codons (e.g., Leu, Ser, Arg) appear more frequently than those with fewer codons (e.g., Met, Trp), matching real experimental library distributions.

## Available tools

In [2]:
display_available_tools("random_protein")

- **`run_random_protein_sample()`** — Sample protein sequences by filling masked positions with random amino acids drawn from a codon scheme

### `run_random_protein_sample`

`run_random_protein_sample` fills masked positions in protein sequences with amino acids drawn from a codon scheme distribution. Positions marked with `_` are replaced by a randomly sampled amino acid; all other positions are preserved unchanged. The codon scheme controls which amino acids are possible and how frequently each appears: UNIFORM weights all 20 amino acids equally, while library-derived schemes like NNK and NDT reflect the codon redundancy of real degenerate oligonucleotide libraries. The masking strategy system lets you select positions by count, fraction, or index, or you can pre-mask positions manually.

In [3]:
from proto_tools import RandomProteinSampleInput, RandomProteinSampleConfig, run_random_protein_sample
from proto_tools.transforms.masking import MaskingStrategy

In [4]:
# Display input docs
display_api_reference("random_protein", "input", "run_random_protein_sample")

# Underscore characters mark positions for random substitution
inputs = RandomProteinSampleInput(sequences=["MK_AY_AKQR"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [5]:
# Display config docs
display_api_reference("random_protein", "config", "run_random_protein_sample")

# Use a masking strategy to automatically select 3 positions to mutate | see docs above for all fields
config = RandomProteinSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=3),
    seed=42,
)

**Config** — `RandomProteinSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `masking_strategy` | `proto_tools.transforms.masking.base.MaskingStrategy` | `MaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None, temperature=1.0, model_name=None, model_checkpoint=None)` | Controls which positions to mask for sampling. Default: random 30%. |
| `codon_scheme` | `Literal['UNIFORM', 'NNN', 'NNK', 'NNS', 'NDT', 'DBK', 'NRT']` | `'UNIFORM'` | Codon scheme for amino acid sampling probabilities. |

In [6]:
# Run the sampling function
wild_type = "MKTLAYAKQR"
result = run_random_protein_sample(
    RandomProteinSampleInput(sequences=[wild_type]),
    config,
)

In [7]:
# Display output docs
display_api_reference("random_protein", "output", "run_random_protein_sample")

# Show the variant and which positions were mutated
variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type:         {wild_type}")
print(f"Variant:           {variant}")
print(f"Mutated positions: {diffs}")

# Demonstrate pre-masked input
result_premask = run_random_protein_sample(RandomProteinSampleInput(sequences=["MK_AY_AKQR"]))
print(f"\nPre-masked input:  MK_AY_AKQR")
print(f"Variant:           {result_premask.sequences[0]}")

# Demonstrate codon scheme distributions
print("\nCodon scheme comparison (all-masked sequence):")
for scheme in ["UNIFORM", "NNK", "NDT"]:
    scheme_config = RandomProteinSampleConfig(codon_scheme=scheme, seed=42)
    scheme_result = run_random_protein_sample(
        RandomProteinSampleInput(sequences=["____"]),
        scheme_config,
    )
    print(f"  {scheme:>7s}: {scheme_result.sequences[0]}")

**Output** — `RandomProteinSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequences with masked positions randomly filled |

Wild type:         MKTLAYAKQR
Variant:           MKTPAYAAQG
Mutated positions: [3, 7, 9]

Pre-masked input:  MK_AY_AKQR
Variant:           MKNAYGAKQR

Codon scheme comparison (all-masked sequence):
  UNIFORM: PAGF
      NNK: EKSR
      NDT: GNHI


## Export Results

Sampling results can be saved to FASTA, TXT, or JSON format for downstream analysis or synthesis ordering.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generate a batch of 5 variants from the same wild-type sequence
batch_inputs = RandomProteinSampleInput(sequences=["MKTLAYAKQR"] * 5)
batch_config = RandomProteinSampleConfig(masking_strategy=MaskingStrategy(num_mutations=2))
batch_result = run_random_protein_sample(batch_inputs, batch_config)

batch_result.export(name="random_protein_variants", export_path=output_dir, file_format="fasta")
print(f"Exported sequences to {output_dir / 'random_protein_variants.fasta'}")

Exported sequences to example_output/random_protein_variants.fasta
